Frequent Itemsets

In [1]:
transactions = [
  ["Bread", "Butter", "Milk"],
  ["Bread", "Butter", "Eggs"],
  ["Milk", "Cereal"],
  ["Bread", "Eggs"],
  ["Milk", "Cereal", "Yogurt"],
  ["Diapers", "Beer"],
  ["Diapers", "Beer", "Chips"],
  ["Apples", "Bananas"],
  ["Apples", "Bananas", "Milk"],
  ["Bread", "Butter", "Milk", "Eggs"],
  ["Cereal", "Milk"],
  ["Bread", "Butter", "Milk", "Eggs"],
  ["Apples", "Milk"],
  ["Diapers", "Beer", "Bread", "Eggs"],
  ["Yogurt", "Cereal"],
  ["Bread", "Butter", "Bananas", "Milk"],
  ["Milk", "Eggs"],
  ["Apples", "Bananas", "Yogurt"],
  ["Bread", "Butter", "Cereal", "Milk"],
  ["Chips", "Salsa"]
]

In [ ]:
def support(candidate_itemset, transactions):
  count = sum(1 for t in transactions if set(candidate_itemset).issubset(set(t)))
  return count / len(transactions)

In [ ]:
def confidence(smaller_trans, larger_trans, transactions):
  # Find the support of the larget set of transactions (X)
  support_denom = support(smaller_trans, transactions)
  if support_denom == 0:
    return 0
  # Find support of the more specific list of transactions (X and Y)
  support_numerator = support(smaller_trans + larger_trans, transactions)
  return support_numerator / support_denom

In [2]:
def apriori(min_supp_threshold, transactions):
  from itertools import combinations

  candidates = []
  k = 1

  # Start with frequent 1-itemsets
  all_items = sorted(set(item for t in transactions for item in t))
  frequent = [frozenset([item]) for item in all_items if support([item], transactions) >= min_supp_threshold]

  candidates.extend(frequent)

  while True:
    k += 1

    # Join frequent (k-1)-length itemsets to generate candidates for length k
    new_candidates = set()
    freq_list = list(frequent)
    # Come up with all subsets
    for i in range(len(freq_list)):
      for j in range(i + 1, len(freq_list)):
        union = freq_list[i] | freq_list[j]
        if len(union) == k:
          new_candidates.add(union)

    # Prune the k-length candidates for which not all (k-1)-length subsets are frequent
    pruned = []
    for candidate in new_candidates:
      # Remove one item from the candidates each time and check if it's frequent
      subsets = [candidate - frozenset([item]) for item in candidate]
      # If all subsets are frequent, keep it
      if all(s in frequent for s in subsets):
        pruned.append(candidate)
      # Otherwise, report which candidates were pruned this way and do not add it
      else:
        print(f"Pruned: {set(candidate)} (not all {k-1}-length subsets are frequent)")

    # Count remaining candidates that pass min_supp
    frequent = [c for c in pruned if support(list(c), transactions) >= min_supp_threshold]

    # If we have no more frequent itemsets, stop
    if not frequent:
      break

    # Add them to our total candidates
    candidates.extend(frequent)

  return [set(c) for c in candidates]

Association Rules

In [ ]:
def apriori_rules(frequent_itemsets, min_conf, transactions):
  rules = []

  # Go through each frequent itemset
  for itemset in frequent_itemsets:
    itemset = frozenset(itemset)
    # Association rules need 2 items
    if len(itemset) < 2:
      continue

    # Start with 1-item consequents (RHS)
    # These are the rules with the largest antecedent (LHS) (e.g., ABC -> D)
    # which have the LOWEST confidence due to anti-monotonicity
    consequents = [frozenset([item]) for item in itemset]

    m = 1
    while len(consequents) > 0 and len(itemset) > m:
      # Count: check which rules meet min_conf
      passing = []
      for consequent in consequents:
        # Remove the consequent from the itemset to create the antecedent
        antecedent = itemset - consequent
        # Compute the comfidence and check if it's greater than min_conf
        conf = confidence(list(antecedent), list(consequent), transactions)
        if conf >= min_conf:
          # Keep track of passing consequents and create a new rule
          passing.append(consequent)
          rules.append((set(antecedent), set(consequent), conf))
        else:
          print(f"Pruned rule: {set(antecedent)} -> {set(consequent)} (conf={conf:.4f} < {min_conf})")

      # Increase the number of consequents, stopping when we can no longer make an antecedent of at least 1
      m += 1
      if m >= len(itemset):
        break

      # Join: merge passing consequents that we just got to get new m-item consequents to try
      potential_new_consequents = set()
      passing_list = list(passing)
      # Any passing consequents have their unions become a new candidate
      for i in range(len(passing_list)):
        for j in range(i + 1, len(passing_list)):
          union = passing_list[i] | passing_list[j]
          if len(union) == m:
            potential_new_consequents.add(union)

      # Prune: only keep if all (m-1)-subsets of the consequent are in passing
      consequents = []
      for candidate in potential_new_consequents:
        subsets = [candidate - frozenset([item]) for item in candidate]
        # Check if new unions we created do not pass
        if all(s in passing for s in subsets):
          consequents.append(candidate)
        else:
          antecedent = itemset - candidate
          print(f"Pruned rule: {set(antecedent)} -> {set(candidate)} (not all sub-consequents passed)")

  return rules

Hierarchical Clustering